# COMP34812 Coursework - Cross Encoder

## Install dependency

In [ ]:
from pathlib import Path
root_path = Path().resolve().parent.parent
print(f"root_path: {root_path}")

# For non-Kaggle working environment
# !pip install -r {root_path}/requirements.txt
# For Kaggle working environment
!pip install matplotlib pandas sentence_transformers torch transformers

print(f"pip install is finished ...")

## Import dependency

In [ ]:
from itertools import product
import math
import matplotlib.pyplot as plt
import os
import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

print(f"Import is finished ...")

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset


class AVDataset(Dataset):
    def __init__(self, csv_file_normal: str, csv_file_manual_stylometry: str, text_1_column="text_1", text_2_column="text_2", author_score_column="author_score", style_similarity_score_column="style_similarity_score", separation_token="[SEP]"):
        # Load CSV
        self.df_normal = pd.read_csv(csv_file_normal)
        self.df_manual_stylomtry = pd.read_csv(csv_file_manual_stylometry)

        assert len(self.df_normal) == len(self.df_manual_stylomtry), f"Mismatch between lengths of dataframes: {len(self.df_normal)} != {len(self.df_manual_stylomtry)}"

        # Columns
        self.text_1 = self.df_normal[text_1_column].astype(str).tolist()
        self.text_2 = self.df_normal[text_2_column].astype(str).tolist()
        if author_score_column in self.df_normal.columns:
            # Non-testing
            self.author_score = self.df_normal[author_score_column].astype(float).tolist()
        else:
            # Testing
            self.author_score = None
        self.style_similarity_score = self.df_normal[style_similarity_score_column].astype(float).tolist()

        # Load the whole csv file
        self.manual_stylomtries = self.df_manual_stylomtry.values

        assert len(self.df_normal) == len(self.manual_stylomtries), f"Mismatch between lengths between self.df_normal and self.manual_stylometries: {len(self.df_normal)} != {len(self.manual_stylomtries)}"

        self.separation_token = separation_token

        # Sanity check
        if self.author_score is not None:
            # Non-testing
            assert len(self.text_1) == len(self.author_score) and len(self.text_2) == len(self.author_score) and len(self.style_similarity_score) == len(self.author_score), f"Mismatch between lengths: text_1={len(self.text_1)}, text_2={len(self.text_2)}, author_score={len(self.author_score)}, style_similarity_score={len(self.style_similarity_score)}"
        else:
            # Testing
            assert len(self.text_1) == len(self.text_2) and len(self.style_similarity_score) == len(self.text_2), f"Mismatch between lengths: text_1={len(self.text_1)}, text_2={len(self.text_2)} style_similarity_score={len(self.style_similarity_score)}"

    def __len__(self):
        return len(self.text_1)

    def __getitem__(self, idx):
        """
        batch_manual_stylometry_y is literally author_score, which is the label
        """
        # Return a tuple
        combined_text = (self.text_1[idx], self.text_2[idx])
        if self.author_score is not None:
            # Non-testing
            author_score = torch.tensor(self.author_score[idx], dtype=torch.float)
        else:
            author_score = None
        style_similarity_score = torch.tensor(
            self.style_similarity_score[idx], dtype=torch.float)
        manual_stylometry = torch.tensor(self.manual_stylomtries[idx], dtype=torch.float)        

        if author_score is not None:
            # Non-testing
            return (combined_text, author_score, style_similarity_score, manual_stylometry, author_score)
        else:
            # Testing
            return (combined_text, style_similarity_score, manual_stylometry)


In [ ]:
import torch
from torch import nn, functional as F
from transformers import AutoModel

"""
Return different scores, which will be used to determine the final score in the task of Author Verification
"""


class MultiHeadCrossEncoder(nn.Module):
    model_name: str
    head_weights: torch.Tensor

    def __init__(self, model_name: str, head_weights: torch.Tensor, manual_stylometry_dim: int = 171):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)

        encoder_hidden_size = self.encoder.config.hidden_size

        self.shared = nn.Linear(encoder_hidden_size, encoder_hidden_size)
        self.shared_activation = nn.ReLU()

        self.head_author = nn.Linear(encoder_hidden_size, 1)
        self.head_style_similarity = nn.Linear(encoder_hidden_size, 1)
        self.head_manual_stylometry = nn.Sequential(
            nn.Linear(manual_stylometry_dim, 64), 
            nn.ReLU(), 
            nn.Linear(64, 1)
        )

        assert isinstance(
            head_weights, torch.Tensor), f"Invalid type of head_weights: {type(head_weights)}"
        assert head_weights.dim() == 1 and head_weights.numel() == 3, f"Invalid shape of head_weights (expect [3]): {head_weights.shape}"

        if sum(head_weights) == 1:
            self.head_weights = head_weights
        else:
            # Apply softmax function
            self.head_weights = F.softmax(head_weights, dim=0)

    def forward(self, input_ids, attention_mask, manual_stylometry_vector: torch.Tensor, token_type_ids=None):
        encoder_outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        embedding_cls = encoder_outputs.last_hidden_state[:, 0]

        shared_output = self.shared_activation(self.shared(embedding_cls))

        # Individual heads
        score_author = self.head_author(shared_output)
        score_style_similarity = self.head_style_similarity(shared_output)
        score_manual_stylometry = self.head_manual_stylometry(manual_stylometry_vector)

        score_final = (self.head_weights[0] * score_author) + (self.head_weights[1] * score_style_similarity) + (self.head_weights[2] * score_manual_stylometry)

        return {
            f"score_final": score_final,
            f"score_author": score_author,
            f"score_style_similarity": score_style_similarity, 
            f"score_manual_stylometry": score_manual_stylometry
        }


## Hyperparameter

In [ ]:
HYPERPARAMETERS_HEAD_WEIGHTS = [
    torch.tensor((0.0, 0.0, 1.0), dtype=torch.float), 
    torch.tensor((0.0, 0.25, 0.75), dtype=torch.float), 
    torch.tensor((0.0, 0.5, 0.5), dtype=torch.float), 
    torch.tensor((0.0, 0.75, 0.25), dtype=torch.float), 
    torch.tensor((0.0, 1.0, 0.0), dtype=torch.float), 
    torch.tensor((0.25, 0.0, 0.75), dtype=torch.float), 
    torch.tensor((0.25, 0.25, 0.5), dtype=torch.float), 
    torch.tensor((0.25, 0.5, 0.25), dtype=torch.float), 
    torch.tensor((0.25, 0.75, 0.0), dtype=torch.float), 
    torch.tensor((0.5, 0.0, 0.5), dtype=torch.float), 
    torch.tensor((0.5, 0.25, 0.25), dtype=torch.float), 
    torch.tensor((0.5, 0.5, 0.0), dtype=torch.float), 
    torch.tensor((0.75, 0.0, 0.25), dtype=torch.float), 
    torch.tensor((0.75, 0.25, 0.0), dtype=torch.float), 
    torch.tensor((1.0, 0.0, 0.0), dtype=torch.float),
]

# The name of the model that both tokenizer and the embedder use (must use the same)
# Consider MODEL_NAME as a specific language used by both tokenizer and the embedder for communication, i.e. they know which token_id represents which token
HYPERPARAMETERS_MODEL_NAME = [
    f"bert-base-uncased", 
    f"bert-base-cased",
]

hyperparameters = list(product(HYPERPARAMETERS_HEAD_WEIGHTS, HYPERPARAMETERS_MODEL_NAME))

## Global variable

In [ ]:
# If prediction >= SIGMOID_THRESHOLD, then it is set to 1. 0 otherwise. 
SIGMOID_THRESHOLD = 0.5

SEPARATION_TOKEN = f"[SEP]"
# Larger batch for efficiency
BATCH_SIZE = 32
SHUFFLE_DATA = False
TRAIN_EPOCH = 20
LEARNING_RATE = 2e-5
# Number of train_epoch to log once
EPOCH_PER_LOG = 1

ROOT_PATH = os.path.dirname(os.path.dirname(os.getcwd()))

# TODO: Update me if applicable
KAGGLE_DATASET_NAME = f"authorverification6"

# TODO: Update me when applicable
CSV_NAME_TRAIN_NORMAL = f"train_final.csv"
# CSV_NAME_TRAIN_NORMAL = f"AV_trial_final.csv"
# For Kaggle working environment
CSV_PATH_TRAIN_NORMAL = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_TRAIN_NORMAL)

CSV_NAME_VALIDATION_NORMAL = f"dev_final.csv"
# For Kaggle working environment
CSV_PATH_VALIDATION_NORMAL = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_VALIDATION_NORMAL)

CSV_NAME_TEST_NORMAL = f"test_final.csv"
# For Kaggle working environment
CSV_PATH_TEST_NORMAL = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_TEST_NORMAL)

# TODO: Update me when applicable
CSV_NAME_TRAIN_MANUAL_STYLOMETRY = f"train_style_diff.csv"
# CSV_NAME_TRAIN_MANUAL_STYLOMETRY = f"trial_style_diff.csv"
# For Kaggle working environment
CSV_PATH_TRAIN_MANUAL_STYLOMETRY = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_TRAIN_MANUAL_STYLOMETRY)

CSV_NAME_VALIDATION_MANUAL_STYLOMETRY = f"val_style_diff.csv"
# For Kaggle working environment
CSV_PATH_VALIDATION_MANUAL_STYLOMETRY = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_VALIDATION_MANUAL_STYLOMETRY)

CSV_NAME_TEST_MANUAL_STYLOMETRY = f"test_style_diff.csv"
# For Kaggle working environment
CSV_PATH_TEST_MANUAL_STYLOMETRY = os.path.join(ROOT_PATH, "kaggle", "input", "datasets", "toothless7788", KAGGLE_DATASET_NAME, CSV_NAME_TEST_MANUAL_STYLOMETRY)

CSV_NAME_OUTPUT = f"predictions.csv"
# For Kaggle working environment
CSV_PATH_OUTPUT = os.path.join(ROOT_PATH, "kaggle", "working", CSV_NAME_OUTPUT)

CROSS_ENCODER_NAME = f"cross-encoder.pth"
# For Kaggle working environment
CROSS_ENCODER_PATH = os.path.join(ROOT_PATH, "kaggle", "working", CROSS_ENCODER_NAME)

print(f"Global variables are set ...")

## Utils

In [ ]:
def log(text=None) -> None:
    if text is None:
        print(f"[INFO]")
    else:
        print(f"[INFO] {text}")

## Prepare data

In [ ]:
log(f"Reading csv file at {CSV_PATH_TRAIN_NORMAL} ...")
    
# Load dataset
# Training
dataset_train = AVDataset(CSV_PATH_TRAIN_NORMAL, CSV_PATH_TRAIN_MANUAL_STYLOMETRY)
dataloader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=SHUFFLE_DATA)
# Validation
dataset_validation = AVDataset(CSV_PATH_VALIDATION_NORMAL, CSV_PATH_VALIDATION_MANUAL_STYLOMETRY)
dataloader_validation = DataLoader(dataset_validation, batch_size=BATCH_SIZE, shuffle=SHUFFLE_DATA)
# Testing
dataset_test = AVDataset(CSV_PATH_TEST_NORMAL, CSV_PATH_TEST_MANUAL_STYLOMETRY)
dataloader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"All data finish preparing ...")

## Instantiate entity

In [ ]:
device = torch.device(f"cuda" if torch.cuda.is_available() else f"cpu")
log(f"Using device {device} ...")

# This expects raw numbers (logits) and handles the Sigmoid internally
criterion = nn.BCEWithLogitsLoss()

print(f"All instances have been instantiated ...")

## Hyperparameter Selection (Training + Validation)
- Note that this will also calculate the testing loss of the model

In [ ]:
"""
In the form of {(head_weights, model_name): {
    "losses_train": dict, 
    "losses_validation": dict
}}
"""
experiment_outputs = {}

experiment_index = 0

for head_weights, model_name in hyperparameters:
    log()
    log(f"Running Experiment [{experiment_index + 1} / {len(hyperparameters)}]")
    log(f"head_weights: {head_weights}")
    log(f"model_name: {model_name}")
    experiment_index += 1

    # Instantiate relevant models
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    cross_encoder = MultiHeadCrossEncoder(
        model_name=model_name, 
        head_weights=head_weights
    ).to(device)
    optimizer = AdamW(cross_encoder.parameters(), lr=LEARNING_RATE)

    experiment_losses_train = {
        f"loss": [], 
        f"loss_author": [], 
        f"loss_style_similarity": [],
        f"loss_manual_stylometry": []
    }
    experiment_losses_validation = {
        f"loss": [], 
        f"loss_author": [], 
        f"loss_style_similarity": [],
        f"loss_manual_stylometry": [], 
        f"f1": [], 
        f"precision": [], 
        f"recall": []
    }


    for epoch in range(TRAIN_EPOCH):
        # =============================== Training ===============================
        metrics_validation = {
            # True positive
            f"tp": 0, 
            # False positive
            f"fp": 0, 
            # True negative
            f"tn": 0, 
            # False negative
            f"fn": 0
        }

        cross_encoder.train()

        average_loss_train = 0.0
        average_loss_train_author = 0.0
        average_loss_train_style_similarity = 0.0
        average_loss_train_manual_stylometry = 0.0

        for batch_X, batch_author_score, batch_style_similarity_score, batch_manual_stylometry_X, batch_manual_stylometry_y in dataloader_train:
            batch_author_score = batch_author_score.to(device)
            batch_style_similarity_score = batch_style_similarity_score.to(device)
            batch_manual_stylometry_X = batch_manual_stylometry_X.to(device)
            batch_manual_stylometry_y = batch_manual_stylometry_y.to(device)
        
            # Reset optimizer for the current gradient descent
            optimizer.zero_grad()

            """
            {
                "input_ids": tensor, 
                "token_type_ids": tensor, 
                "attention_mask": tensor
            }
            """
            encoding_X = tokenizer(
                batch_X[0], 
                batch_X[1], 
                padding=True, 
                truncation=True, 
                return_tensors=f"pt"
            ).to(device)

            y_predictions = cross_encoder(
                input_ids=encoding_X["input_ids"], 
                attention_mask=encoding_X["attention_mask"],
                token_type_ids=encoding_X.get("token_type_ids"),
                manual_stylometry_vector=batch_manual_stylometry_X
            )

            # Calculate individual losses
            loss_train_author = criterion(y_predictions["score_author"].view(-1), batch_author_score)
            loss_train_style_similarity = criterion(y_predictions["score_style_similarity"].view(-1), batch_style_similarity_score)
            loss_train_manual_stylomtry = criterion(y_predictions["score_manual_stylometry"].view(-1), batch_manual_stylometry_y)

            # Combine loss
            loss_train = head_weights[0] * loss_train_author + head_weights[1] * loss_train_style_similarity+  head_weights[2] * loss_train_manual_stylomtry

            average_loss_train += loss_train.item()
            average_loss_train_author += loss_train_author.item()
            average_loss_train_style_similarity += loss_train_style_similarity.item()
            average_loss_train_manual_stylometry += loss_train_manual_stylomtry.item()

            # Apply gradient descent
            loss_train.backward()
            optimizer.step()


        # Update data to be plotted on graphs
        experiment_losses_train[f"loss"].append(average_loss_train / len(dataloader_train))
        experiment_losses_train[f"loss_author"].append(average_loss_train_author / len(dataloader_train))
        experiment_losses_train[f"loss_style_similarity"].append(average_loss_train_style_similarity / len(dataloader_train))
        experiment_losses_train[f"loss_manual_stylometry"].append(average_loss_train_manual_stylometry / len(dataloader_train))
        
        if epoch % EPOCH_PER_LOG == 0:
            # Log
            log(f"[EPOCH {epoch + 1} / {TRAIN_EPOCH}] Training loss: {experiment_losses_train[f'loss'][-1]}")
        
        # =============================== Testing ===============================
        cross_encoder.eval()

        average_loss_validation = 0.0
        average_loss_validation_author = 0.0
        average_loss_validation_style_similarity = 0.0
        average_loss_validation_manual_stylometry = 0.0

        with torch.no_grad():
            for batch_X, batch_author_score, batch_style_similarity_score, batch_manual_stylometry_X, batch_manual_stylometry_y in dataloader_validation:
                batch_author_score = batch_author_score.to(device)
                batch_style_similarity_score = batch_style_similarity_score.to(device)
                batch_manual_stylometry_X = batch_manual_stylometry_X.to(device)
                batch_manual_stylometry_y = batch_manual_stylometry_y.to(device)

                """
                {
                    "input_ids": tensor, 
                    "token_type_ids": tensor, 
                    "attention_mask": tensor
                }
                """
                encoding_X = tokenizer(
                    batch_X[0], 
                    batch_X[1], 
                    padding=True, 
                    truncation=True, 
                    return_tensors=f"pt"
                ).to(device)

                y_predictions = cross_encoder(
                    input_ids=encoding_X["input_ids"], 
                    attention_mask=encoding_X["attention_mask"],
                    token_type_ids=encoding_X.get("token_type_ids"),
                    manual_stylometry_vector=batch_manual_stylometry_X
                )

                # Calculate individual losses
                loss_validation_author = criterion(y_predictions["score_author"].detach().view(-1), batch_author_score)
                loss_validation_style_similarity = criterion(y_predictions["score_style_similarity"].detach().view(-1), batch_style_similarity_score)
                loss_validation_manual_stylomtry = criterion(y_predictions["score_manual_stylometry"].detach().view(-1), batch_manual_stylometry_y)

                # Combine loss
                loss_validation = head_weights[0] * loss_validation_author + head_weights[1] * loss_validation_style_similarity + head_weights[2] * loss_validation_manual_stylomtry

                average_loss_validation += loss_validation.item()
                average_loss_validation_author += loss_validation_author.item()
                average_loss_validation_style_similarity += loss_validation_style_similarity.item()
                average_loss_validation_manual_stylometry += loss_validation_manual_stylomtry.item()

                # Update metrics
                # Shape: [batch_size]
                y_prediction_final = torch.sigmoid(y_predictions["score_final"].detach().view(-1))

                labels_predicted = y_prediction_final.tolist()
                labels_actual = batch_author_score.tolist()

                assert len(labels_predicted) == len(labels_actual), f"Mismatch between lengths of labels_predicted and labels_actual: {len(labels_predicted)} != {len(labels_actual)}"

                for i in range(len(labels_predicted)):
                    label_predicted = labels_predicted[i]
                    label_actual = labels_actual[i]

                    if label_actual == 1:
                        # True positive/false negative
                        if label_predicted >= SIGMOID_THRESHOLD:
                            # True positive
                            metrics_validation["tp"] += 1
                        else:
                            # False negative
                            metrics_validation["fn"] += 1
                    elif label_actual == 0:
                        # True negative/false positive
                        if label_predicted >= SIGMOID_THRESHOLD:
                            # False positive
                            metrics_validation["fp"] += 1
                        else:
                            # True negative
                            metrics_validation["tn"] += 1
                    else:
                        raise Exception(f"Invalid lable_actual (expect 0 or 1): {label_actual}")

            # Update data to be plotted on graphs
            experiment_losses_validation[f"loss"].append(average_loss_validation / len(dataloader_validation))
            experiment_losses_validation[f"loss_author"].append(average_loss_validation_author / len(dataloader_validation))
            experiment_losses_validation[f"loss_style_similarity"].append(average_loss_validation_style_similarity / len(dataloader_validation))
            experiment_losses_validation[f"loss_manual_stylometry"].append(average_loss_validation_manual_stylometry / len(dataloader_validation))

            # Calculate F1 score
            precision = metrics_validation[f"tp"] / (metrics_validation[f"tp"] + metrics_validation[f"fp"])
            recall = metrics_validation[f"tp"] / (metrics_validation[f"tp"] + metrics_validation[f"fn"])
            f1_score = (2 * precision * recall) / (precision + recall)

            experiment_losses_validation[f"f1"].append(f1_score)
            experiment_losses_validation[f"precision"].append(precision)
            experiment_losses_validation[f"recall"].append(recall)
        
        if epoch % EPOCH_PER_LOG == 0:
            # Log
            log(f"[EPOCH {epoch + 1} / {TRAIN_EPOCH}] Testing loss: {experiment_losses_validation[f'loss'][-1]}")
        
    experiment_output = {
        "losses_train": experiment_losses_train, 
        "losses_validation": experiment_losses_validation
    }
    experiment_outputs[(head_weights, model_name)] = experiment_output

    # Save model weights
    experiment_cross_encoder_path = f"{CROSS_ENCODER_PATH[:-4]}_{head_weights.tolist()}_{model_name}.pth"
    torch.save(cross_encoder.state_dict(), experiment_cross_encoder_path)

    log(f"Model weights are saved to {experiment_cross_encoder_path} ...")

In [ ]:
# Number of graphs
graph_num = len(hyperparameters)

## Plot graph

In [ ]:
# TODO: Update me if you want different number of columns
cols = 3
rows = math.ceil(graph_num / cols)

fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows))
axes = axes.flatten()  # makes indexing easier

### Loss

In [ ]:
for i, (head_weights, model_name) in enumerate(hyperparameters):
    ax = axes[i]
    
    key = f"{head_weights.tolist()}_{model_name}"
    loss_train = experiment_outputs[key]["losses_train"]["loss"]
    loss_validation = experiment_outputs[key]["losses_validation"]["loss"]
    
    epochs = range(1, len(loss_train) + 1)
    
    ax.plot(epochs, loss_train, label="Training")
    ax.plot(epochs, loss_validation, label="Validation")
    
    ax.set_title(f"{model_name}\n{head_weights.tolist()}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(True)
    ax.legend()

    # Clean up empty plots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    # Show graph
    plt.tight_layout()
    plt.show()

### Author Score Loss

In [ ]:
for i, (head_weights, model_name) in enumerate(hyperparameters):
    ax = axes[i]
    
    key = f"{head_weights.tolist()}_{model_name}"
    loss_author_train = experiment_outputs[key]["losses_train"]["loss_author"]
    loss_author_validation = experiment_outputs[key]["losses_validation"]["loss_author"]
    
    epochs = range(1, len(loss_author_train) + 1)
    
    ax.plot(epochs, loss_author_train, label="Training")
    ax.plot(epochs, loss_author_validation, label="Validation")
    
    ax.set_title(f"{model_name}\n{head_weights.tolist()}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Author score")
    ax.grid(True)
    ax.legend()

    # Clean up empty plots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    # Show graph
    plt.tight_layout()
    plt.show()

### Style Similarity Score Loss

In [ ]:
for i, (head_weights, model_name) in enumerate(hyperparameters):
    ax = axes[i]
    
    key = f"{head_weights.tolist()}_{model_name}"
    loss_style_similarity_train = experiment_outputs[key]["losses_train"]["loss_style_similarity"]
    loss_style_similarity_validation = experiment_outputs[key]["losses_validation"]["loss_style_similarity"]
    
    epochs = range(1, len(loss_style_similarity_train) + 1)
    
    ax.plot(epochs, loss_style_similarity_train, label="Training")
    ax.plot(epochs, loss_style_similarity_validation, label="Validation")
    
    ax.set_title(f"{model_name}\n{head_weights.tolist()}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Style similarity score")
    ax.grid(True)
    ax.legend()

    # Clean up empty plots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    # Show graph
    plt.tight_layout()
    plt.show()

### Manual Stylometry Loss

In [ ]:
for i, (head_weights, model_name) in enumerate(hyperparameters):
    ax = axes[i]
    
    key = f"{head_weights.tolist()}_{model_name}"
    loss_manual_stylometry_train = experiment_outputs[key]["losses_train"]["loss_manual_stylometry"]
    loss_manual_stylometry_validation = experiment_outputs[key]["losses_validation"]["loss_manual_stylometry"]
    
    epochs = range(1, len(loss_manual_stylometry_train) + 1)
    
    ax.plot(epochs, loss_manual_stylometry_train, label="Training")
    ax.plot(epochs, loss_manual_stylometry_validation, label="Validation")
    
    ax.set_title(f"{model_name}\n{head_weights.tolist()}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Manual stylometry score")
    ax.grid(True)
    ax.legend()

    # Clean up empty plots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    # Show graph
    plt.tight_layout()
    plt.show()

### F1 Score

In [ ]:
for i, (head_weights, model_name) in enumerate(hyperparameters):
    ax = axes[i]
    
    key = f"{head_weights.tolist()}_{model_name}"
    f1_train = experiment_outputs[key]["losses_train"]["f1"]
    f1_validation = experiment_outputs[key]["losses_validation"]["f1"]
    
    epochs = range(1, len(f1_train) + 1)
    
    ax.plot(epochs, f1_train, label="Training")
    ax.plot(epochs, f1_validation, label="Validation")
    
    ax.set_title(f"{model_name}\n{head_weights.tolist()}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("F1 score")
    ax.grid(True)
    ax.legend()

    # Clean up empty plots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    # Show graph
    plt.tight_layout()
    plt.show()

### Precision

In [ ]:
for i, (head_weights, model_name) in enumerate(hyperparameters):
    ax = axes[i]
    
    key = f"{head_weights.tolist()}_{model_name}"
    precision_train = experiment_outputs[key]["losses_train"]["precision"]
    precision_validation = experiment_outputs[key]["losses_validation"]["precision"]
    
    epochs = range(1, len(precision_train) + 1)
    
    ax.plot(epochs, precision_train, label="Training")
    ax.plot(epochs, precision_validation, label="Validation")
    
    ax.set_title(f"{model_name}\n{head_weights.tolist()}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Precision")
    ax.grid(True)
    ax.legend()

    # Clean up empty plots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    # Show graph
    plt.tight_layout()
    plt.show()

### Recall

In [ ]:
for i, (head_weights, model_name) in enumerate(hyperparameters):
    ax = axes[i]
    
    key = f"{head_weights.tolist()}_{model_name}"
    recall_train = experiment_outputs[key]["losses_train"]["recall"]
    recall_validation = experiment_outputs[key]["losses_validation"]["recall"]
    
    epochs = range(1, len(recall_train) + 1)
    
    ax.plot(epochs, recall_train, label="Training")
    ax.plot(epochs, recall_validation, label="Validation")
    
    ax.set_title(f"{model_name}\n{head_weights.tolist()}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Recall")
    ax.grid(True)
    ax.legend()

    # Clean up empty plots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    # Show graph
    plt.tight_layout()
    plt.show()

## Find Best Hyperparameters

In [ ]:
# Best hyperparameters
HEAD_WEIGHTS = torch.tensor([0.3, 0.3, 0.4], dtype=torch.float), 
MODEL_NAME = f"bert-base-uncased"

best_cross_encoder_path = f"{CROSS_ENCODER_PATH[:-4]}_{HEAD_WEIGHTS.tolist()}_{MODEL_NAME}.pth"

## Run model in inference mode

In [ ]:
cross_encoder = MultiHeadCrossEncoder(model_name=MODEL_NAME, head_weights=HEAD_WEIGHTS).to(device)
cross_encoder.load_state_dict(torch.load(best_cross_encoder_path))

log(f"Cross encoder is loaded from {best_cross_encoder_path} ...")

cross_encoder.eval()

# Will be saved in predictions.csv
predictions = []

with torch.no_grad():
    for batch_X, batch_style_similarity_score, batch_manual_stylometry_X in dataloader_test:
        batch_style_similarity_score = batch_style_similarity_score.to(device)
        batch_manual_stylometry_X = batch_manual_stylometry_X.to(device)

        """
        {
            "input_ids": tensor, 
            "token_type_ids": tensor, 
            "attention_mask": tensor
        }
        """
        encoding_X = tokenizer(
            batch_X[0], 
            batch_X[1], 
            padding=True, 
            truncation=True, 
            return_tensors=f"pt"
        ).to(device)

        y_predictions = cross_encoder(
            input_ids=encoding_X["input_ids"], 
            attention_mask=encoding_X["attention_mask"],
            token_type_ids=encoding_X.get("token_type_ids"),
            manual_stylometry_vector=batch_manual_stylometry_X
        )

        # Shape: [batch_size]
        y_prediction_final = (torch.sigmoid(y_predictions["score_final"].detach().view(-1)) >= SIGMOID_THRESHOLD).int()

        predictions.extend(y_prediction_final.tolist())

log(f"Finish running inference mode ...")

### Generate Predictions CSV File

In [ ]:
df_predictions = pd.DataFrame(predictions, columns=["prediction"])

df_predictions.to_csv(CSV_PATH_OUTPUT, index=False)

print(f"Predictions are saved to {CSV_PATH_OUTPUT} ...")